In [ ]:
import importlib
import torch
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import pathlib
from hydra.core.hydra_config import HydraConfig

model_name = "2D_Fernandes_Phelan"
relative_path = os.path.join('..', '..', 'dptorch')

notebook_dir = os.getcwd()
absolute_path = os.path.abspath(os.path.join(notebook_dir, relative_path))

sys.path.insert(0, absolute_path)

def _iter_num(p):
    try:
        return int(p.stem.split("_")[-1])
    except ValueError:
        return -1
list_all_Iter=list(pathlib.Path(os.path.abspath(f"data/2D")).rglob("*.pth"))
all_checkpoint_iters = [_iter_num(p) for p in list_all_Iter]

latest_checkpoint_num = max(all_checkpoint_iters)
#### what to load
checkpoint_file = latest_checkpoint_num
last_stored_checkpoint = 2999


model = importlib.import_module(f"{model_name}.Model")

# RNG
torch.manual_seed(123)


# load the specific checkpoint
m = model.SpecifiedModel.load(
    path=os.path.abspath(f"data/2D/Iter_{checkpoint_file}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name},
)


sigma = m.cfg["model"]["params"]["sigma"]
beta = m.cfg["model"]["params"]["beta"]
n_types = m.cfg["model"]["params"]["n_types"]
gp_offset = m.cfg["model"]["params"]["GP_offset"]

In [ ]:
m_l2 = float(np.mean((m.metrics[list(m.metrics.keys())[-1]]["l2"]).numpy()))
m_inf = float(np.mean((m.metrics[list(m.metrics.keys())[-1]]["l_inf"]).numpy()))

table_path = os.path.join(notebook_dir, "2D_FP_pointwise_error_table.tex")
with open(table_path, "w", encoding="utf-8") as f:
    f.write(
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\begin{tabular}{lcc}\n"
        "\\hline\n"
        "2D Model & $L_2$ & $L_\\infty$ \\\\\n"
        "\\hline\n"
        f"Baseline & ${m_l2:.4e}$ & ${m_inf:.4e}$ \\\\\n"
        "\\hline\n"
        "\\end{tabular}\n"
        "\\end{table}\n"
    )

print(f"Saved LaTeX table to: {table_path}")

In [ ]:
max_vec = np.zeros(n_types)
for i in range(n_types):
    mask_state = m.state_sample_all[:,n_types] == i
    max_vec[i] = np.max((m.combined_sample_all[mask_state][:,0]).numpy()) + gp_offset

print("max_vec:", max_vec)

### Feasible Set

In [ ]:
points = np.loadtxt((f"data/2D/simulation_{checkpoint_file}.txt"))
VF_plot_data = np.loadtxt((f"data/feas_set/V_func_all_314_0.txt"))

disc_states = (points[:,2]).astype(np.int64)
mask_0 = disc_states == 0
mask_1 = disc_states == 1
# Scale points to [0, 10]x[0, 10] to match original paper
points_0 = points[mask_0]* (1 - sigma) / (1 - beta)
points_1 = points[mask_1]* (1 - sigma) / (1 - beta)
x_scatter = points_0[:, 0]
y_scatter = points_0[:, 1]

x = (VF_plot_data[:,0])
y = (VF_plot_data[:,1])
from scipy.ndimage import gaussian_filter
z = VF_plot_data[:,2]

from scipy.interpolate import griddata

grid_x, grid_y = np.meshgrid(np.linspace(x.min(), x.max(), 1000), 
                             np.linspace(y.min(), y.max()-0.075, 1000))

grid_z = griddata((x.ravel(), y.ravel()), z.ravel(), (grid_x, grid_y), method='cubic')

  

contour_level = -0.04
plt.contourf(grid_x, grid_y, grid_z, levels=[-np.inf, contour_level], colors='red', alpha=0.5) 
plt.contourf(grid_x, grid_y, grid_z, levels=[contour_level, np.inf], colors='blue', alpha=0.5) 
plt.contour(grid_x, grid_y, grid_z, levels=[contour_level], colors='black', linewidths=2, linestyles='solid')
plt.scatter(x_scatter, y_scatter, marker='o', color='green', label='Simulation')
plt.grid(True)

# plt.xticks(np.linspace(0, 10, 11))
# plt.yticks(np.linspace(0, 10, 11))

plt.xlim(1.5 ,9.5 )
plt.ylim(3. ,10. )

blue_patch = plt.Line2D([0], [0], color='blue', lw=4, label='Feasible')
red_patch = plt.Line2D([0], [0], color='red', lw=4, label='Infeasible')
plt.legend(handles=[blue_patch, red_patch], loc='lower right', fontsize=14)

plt.xlabel(r"$\mathbf{v_1}$", fontsize=14)
plt.ylabel(r"$\mathbf{v_2}$", fontsize=14)

plt.savefig('Figure_5_feas_state_with_sim.pdf', dpi=400)

### BAL error plot

In [ ]:
def mean_squared_error_GP(model, eval_pt, discrete_state, target_p):

    #compute the mean squared error MSE according to Dario Azzimonti Gaussian processes and sequential design of experiments (lecture)
    if eval_pt.shape[0] > 1:             
        eval_pt = torch.unsqueeze(eval_pt,1) #doing a batch of points needs to be done as batches of single points

    train_inputs = model.M[discrete_state][target_p].train_inputs[0]

    kxx = model.M[discrete_state][target_p].covar_module(eval_pt).evaluate()
    kXx = model.M[discrete_state][target_p].covar_module(train_inputs,eval_pt).evaluate()
    kXX = model.M[discrete_state][target_p].covar_module(train_inputs,train_inputs).evaluate()
    try:
        U = torch.linalg.cholesky(kXX)
    except:
        U = torch.linalg.cholesky(kXX + 1e-2*torch.eye(kXX.shape[0]))
    kXXinv_kXx = torch.cholesky_solve(kXx, U)
    out_vec_ = kxx - torch.tensordot(kXx.transpose(-1,-2),kXXinv_kXx,[[-2,-1],[-1,-2]])

    out_vec = torch.diagonal(out_vec_)
    if out_vec.ndim == 2:
        out_vec = torch.diag(out_vec)

    return out_vec

In [ ]:
mask_0 = m.state_sample[:,-1] == 0.
no_init_samples = m.cfg["no_samples"]
n_pts = m.state_sample[mask_0,:].shape[0]

bal_util = torch.zeros(n_pts-no_init_samples)
for indxp in range(n_pts-no_init_samples):
    train_sample = m.state_sample[mask_0,:][:no_init_samples+indxp,:-1]
    train_v = m.V_sample[mask_0][:no_init_samples+indxp]
    m.M[0][0].set_train_data(
        train_sample,
        train_v,
        strict=False,
    )

    bal_util[indxp] = mean_squared_error_GP(m,m.state_sample[mask_0,:][no_init_samples+indxp:no_init_samples+indxp+1,:-1],0,0)

In [ ]:
points = m.state_sample
disc_states = (points[:,2])
mask_0 = disc_states == 0
mask_1 = disc_states == 1

points_0 = points[mask_0]* (1 - sigma) / (1 - beta)
points_1 = points[mask_1]* (1 - sigma) / (1 - beta)

points_set_1 = points_0[:32,:]
points_set_2 = points_0[32:114,:]
points_set_3 = points_0[114:,:]




error_data= np.loadtxt(f"data/2D/V_func_error_0_{last_stored_checkpoint}.txt")
T1 = error_data[:,0]
T2 = (error_data[:,1]).astype(int)
L2 = error_data[:,2]
Linf = error_data[:,3]
simerror_L2 = error_data[:,4]
simerror_Linf = error_data[:,5]
data_x = np.arange(bal_util.shape[0])
indx_before = 28 + np.arange(0,30*(bal_util.shape[0]+1),30)

figs, axs = plt.subplots(1,3,figsize=(20, 5)) # Adjust figure size as needed

axs[0].plot(
    data_x*30,
    bal_util.detach().numpy(),
    color='tab:blue',  
    linestyle='-',     
    marker='o',        
)

axs[0].set_xlabel('Iteration steps', fontsize=14, fontweight='bold')
axs[0].set_ylabel(r'$\sigma^2$', fontsize=14, fontweight='bold')

axs[0].grid(
    True, 
    linestyle='--', # Dashed line style
    alpha=0.6      # Transparency
)

from scipy.ndimage import gaussian_filter

# Apply Gaussian filter for smoothing
smooth_L2 = gaussian_filter(np.log(L2)/np.log(10), sigma=10.)
smooth_Linf = gaussian_filter(np.log(Linf)/np.log(10), sigma=10.)
smooth_simerror_L2 = gaussian_filter(np.log(simerror_L2)/np.log(10), sigma=10.)
smooth_simerror_Linf = gaussian_filter(np.log(simerror_Linf)/np.log(10), sigma=10.)

axs[1].plot(
    T2,
    smooth_L2,
    color='tab:blue',  
    linestyle='-',
    label='L2 Error'
)

axs[1].plot(
    T2,
    smooth_Linf,
    color='tab:orange',  
    linestyle='--',
    label='Linf Error'
)

axs[1].plot(
    T2,
    smooth_simerror_L2,
    color='tab:green',  
    linestyle=':',
    label='L2 Simulation Error'
)

axs[1].plot(
    T2,
    smooth_simerror_Linf,
    color='tab:red',  
    linestyle='-.',
    label='Linf Simulation Error'
)

axs[1].set_xlabel('Iterations', fontsize=14, fontweight='bold')
axs[1].set_ylabel('Log Error', fontsize=14, fontweight='bold')
axs[1].legend(fontsize=12)
axs[1].grid(True, linestyle='--', alpha=0.6)

# Plot each set of points with different markers and line styles
axs[2].scatter(points_set_1[:, 0], points_set_1[:, 1], marker='o', edgecolor='black', label='Random')
axs[2].scatter(points_set_2[:, 0], points_set_2[:, 1], marker='s', edgecolor='black', label='BAL first 50')
axs[2].scatter(points_set_3[:, 0], points_set_3[:, 1], marker='^', edgecolor='black', label='BAL all other')

# Add labels and legend
axs[2].set_xlabel(r"$\mathbf{v_1}$", fontsize=14)
axs[2].set_ylabel(r"$\mathbf{v_2}$", fontsize=14)
axs[2].legend()


plt.savefig('Figure_4_BAL_Acquisition_Convergence.pdf', dpi=400)

### Error table

In [ ]:
if last_stored_checkpoint == checkpoint_file:
    error_data= np.loadtxt(f"data/2D/V_func_error_0_{latest_checkpoint_num}.txt")
else:
    error_data= np.loadtxt(f"data/2D/V_func_error_{latest_checkpoint_num}.txt")
L2 = error_data[:,2]
Linf = error_data[:,3]
simerror_L2 = error_data[:,4]
simerror_Linf = error_data[:,5]

table_text = f"""    \\begin{{tabular}}{{l|c|c}}
    \hline \hline 
    \\text{{Error type}} &  \\text{{$L_2$}} & \\text{{$L_\infty$}} \\\\
      \hline \hline
     Criterion 2 (global error) & {L2[-1]:.1e}   & {Linf[-1]:.1e} \\\\
     Criterion 3 (error along a simulated path) & {simerror_L2[-1]:.1e} & {simerror_Linf[-1]:.1e} \\\\
     \hline
    \end{{tabular}}"""

print(table_text)

with open("Table_2_error_table.tex", "w", encoding="utf-8") as table_file:
    table_file.write(table_text)
